# Donut fine-tune - InBody **270 + 570** v6 - Kaggle runner

Retrains Donut on the **v6** both-device synthetic set (2500 `inbody_270` + 2500 `inbody_570`).

**What v6 changes, and what it is testing.** Every bar row used to print its value *on top of*
its own axis tick labels, lifted to `top:-17px` and masked with a white halo. Rendered, the BMI
row read `10 15 [18.3] 21 25 30` with the value having painted out the `18.5` tick entirely
(#52). `percent_body_fat` is the worst exposed and uniquely so: its values run 10-35 against ticks
8..58, so it is the only scored field whose magnitude matches its own axis, and it is always
flanked on both sides.

v6 prints the value on its own line below the ticks, on both devices, with nothing behind it and
so nothing to mask -- which is what a real sheet does. The 570's rows went 32px to 40px to match
the 270 so both clear their ticks by the same 9.2px (#55); that costs 47px of sheet height and
leaves the aspect at 0.694, inside the A4 tolerance.

**Nothing else changed.** No geometry tuning, no augmentation change, no new fields.

### The prediction, written before the run

v5 reads `percent_body_fat` **6/12** on the real hold-out against v3's 12/12. It is the only
field v5 loses on, and the only reason v3 led core fields at all. So:

- if v6 recovers `percent_body_fat` toward 12/12 while holding v5's segmental lean (52/60) and
  critical fields (64/72), the value/tick collision was the cause and this is the engine to ship;
- if `percent_body_fat` stays around 6/12, the collision was not the cause, #48's surviving
  hypothesis is wrong, and the field needs a new explanation rather than another template edit.

Recording it here so v6 is read against a stated expectation rather than in hindsight, which is
how v4 became unattributable.

**Budget the 12 h session cap.** 3 epochs on 5000 sheets completed in well under it for v5
(final loss 0.0026 at step 3750), and `processor.save_pretrained` only runs after
`trainer.train()` returns -- so a processor at the run root is the cheapest proof the run
finished rather than being cut off.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the branch carrying the v5 templates, the manifest and the silent-error scorer.
BRANCH = 'feat/module-1-real-holdout-scorer'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown

In [ ]:
# Dataset attached as a Kaggle Dataset (Add Input, right panel) - no Drive/gdown.
# Find the sheets wherever Kaggle mounted them; if the input is still a .zip,
# extract it to /kaggle/tmp. Sets DATA_DIR to the folder holding the sheets.
import glob, os, shutil

# generate_dataset writes .jpg (inform.training.dataset.IMAGE_SUFFIXES); runs
# before that wrote .png, so accept either rather than silently finding nothing.
SUFFIXES = ('jpg', 'jpeg', 'png')

def find_sheets(root):
    return sorted(p for s in SUFFIXES for p in glob.glob(f'{root}/**/*.{s}', recursive=True))

sheets = find_sheets('/kaggle/input')
if not sheets:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No sheets or zip under /kaggle/input - click Add Input and attach your dataset.'
    shutil.unpack_archive(zips[0], '/kaggle/tmp')
    sheets = find_sheets('/kaggle/tmp')
assert sheets, 'No sheets found after extract - check the dataset contents.'
DATA_DIR = os.path.dirname(sheets[0])
n270 = len([p for p in sheets if 'inbody_270' in p]); n570 = len([p for p in sheets if 'inbody_570' in p])
print('DATA_DIR =', DATA_DIR, '| sheets:', len(sheets), '| 270:', n270, '570:', n570)


In [ ]:
# Fresh from donut-base, 3 epochs, BOTH devices. Do NOT resume from v3, v4 or v5:
# v5 changed the input distribution again (Segmental Fat panel on both devices).
# batch 1 + grad-accum 4 fits the 2560x1920 canvas on a T4; effective batch 4.
# --batch-size 2 OOMs even on 16 GB - raise grad-accum instead.
# 4 dataloader workers keep the GPU fed while JPEGs decode.
# 3 epochs fits the 12 h cap, so the run completes and saves its processor.
CHECKPOINT_DIR = '/kaggle/working/donut-both-v6'
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
  --dataloader-num-workers 4 --learning-rate 3e-5


In [ ]:
# RESUME ONLY - run this instead of cell 4 when a previous session hit the 12 h cap.
# Attach that session's output as an input, copy the checkpoints into CHECKPOINT_DIR,
# then --resume picks up from the last checkpoint-* subdir.
#
# import glob, shutil, os
# prev = glob.glob('/kaggle/input/**/donut-both-v6', recursive=True)[0]
# shutil.copytree(prev, CHECKPOINT_DIR, dirs_exist_ok=True)
# !python -m inform.training.train \
#   --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
#   --model-name-or-path naver-clova-ix/donut-base \
#   --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
#   --dataloader-num-workers 4 --learning-rate 3e-5 --resume


In [ ]:
# Every epoch checkpoint is kept (save_strategy='epoch', save_total_limit=None), so
# /kaggle/working holds checkpoint-* subdirs plus the final top-level model. All of it
# lands in the Save Version output. ~2.4 GB per checkpoint - watch the 20 GB /kaggle/working
# limit, and download to a drive with room (NOT C:, which has ~12 GB free).
!du -sh /kaggle/working/donut-both-v6/* | sort -h
!ls -la /kaggle/working/donut-both-v6


## After the run

Score **every** epoch checkpoint, not just the last.

The hold-out is 12 of 12 hand-labelled as of 2026-09-12, so this is 132 field observations and
the blind spot is closed. Since ADR-0011 the engine crops the sheet out of the photo itself, so
scoring the raw hold-out is correct and a *recorded* `--reads` baseline from before that date is
not comparable -- it replays uncropped.

```bash
python -m inform.holdout --data-dir data/real_holdout \
    --labels data/real_holdout/labels.json \
    --donut-checkpoint /path/to/donut-both-v6/checkpoint-<N>
```

Per-epoch checkpoint dirs carry no processor; copy `processor_config.json`, `tokenizer.json` and
`tokenizer_config.json` from the run root into each one before loading it.

### Baseline to beat: v5-e3750, the current default engine

| n=12, 132 labelled field observations | v3 | v5-e3750 (default) |
|---|---|---|
| core fields | 61/72 | **63/72** |
| segmental lean | 31/60 | **52/60** |
| critical (LBM + limbs) | 43/72 | **64/72** |
| silent errors, sheets | 4/7 57.1% | **1/5 20.0%** |
| silent errors, fields | 4/77 5.2% | **1/55 1.8%** |
| `percent_body_fat` | **12/12** | 6/12 |
| `segmental_lean.right_arm_kg` | 5/12 | **10/12** |

### Read the headline beside the outcome split, always

A silent error is a wrong value inside an `unverified` read: wrong, and carrying no signal that
anything is wrong (CONTEXT.md, ADR-0006's 2026-09-11 amendment). An engine that flags or
under-reads everything has no `unverified` reads and therefore a perfect headline, which is why
the scorer prints both together. Two of v4's four checkpoints scored exactly that way.

Pass several `--reads`/`--donut-checkpoint` for the comparison table. Two runs both name a
checkpoint `checkpoint-3750`; the table disambiguates by path suffix, so v5's and v6's can be
compared directly.
